In [ ]:
%pylab inline

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import pandas as pd
from glob import glob
from tqdm import tqdm
from skimage import io

In [ ]:
from skimage.transform import rotate, resize
from skimage.exposure import equalize_adapthist

In [ ]:
path_images = './00.data/cropped_old/'
path_labels = './RotatedImageFiles.txt'

In [ ]:
data_files = glob(path_images+'*')
data_files_zero = [path for path in data_files if '_0.jpg' in path]
data_labels = pd.read_csv(path_labels)

In [ ]:
data_labels.index = data_labels.Filename.values

In [ ]:
def get_image(img_path):
    
    img = io.imread(img_path)
    
    return(img)


def get_label(img_path,out=data_labels):
    
    img_id = img_path.split('/')[-1]
    row = out.loc[img_id]
    angle = row['Doppler_Angle']
    
    return(angle)

In [ ]:
data_x = [get_image(path) for path in data_files]
data_y = [get_label(path) for path in data_files]

In [ ]:
data_x_zero = [get_image(path) for path in data_files_zero]
data_y_zero = [get_label(path) for path in data_files_zero]

In [ ]:
angle_set = np.arange(-30,35,5)
rotation_angle = np.random.choice(a = angle_set)
image_index = np.random.choice(np.arange(0,len(data_x_zero)))

img_x = data_x_zero[image_index]
img_y = data_y_zero[image_index]
new_angle = img_y+rotation_angle


img_x = resize(img_x,(110,110),mode='reflect')
img_rot = rotate(img_x,angle=rotation_angle,mode='reflect')


print(img_y,rotation_angle,new_angle)

plt.figure(figsize=(10,6))

plt.subplot(131)
plt.imshow(img_x,)
plt.title('Original : '+str(img_y));

plt.subplot(132)
plt.imshow(img_rot)
plt.title('New : Original'+'+'+str(rotation_angle)+' = '+str(new_angle));

plt.subplot(133)
plt.imshow(equalize_adapthist(img_x))
plt.title('Normalized : '+str(img_y));

In [ ]:
img_x = data_x_zero[image_index]

plt.imshow(img_x)
plt.grid('off')
plt.axis('off');

In [ ]:
img_path = data_files_zero[0]
img_id = img_path.split('/')[-1]
img_zero = plt.imread(img_path)

img_cols = 10
rotation_step = 10

i = 1

print(data_labels.loc[img_id]['Doppler_Angle'])
plt.figure(figsize=(15,4))
for ix in range(img_cols):
    angle_pos = ix*rotation_step
    angle_neg = -1*ix*rotation_step
    
    img_rot01 = rotate(img_zero,angle=angle_pos,mode='reflect')
    img_rot02 = rotate(img_zero,angle=angle_neg,mode='reflect')
    
    plt.subplot(2,img_cols,i)
    plt.imshow(img_rot01)
    plt.title(angle_pos)
    plt.axis('off')
    
    plt.subplot(2,img_cols,i+img_cols)
    plt.imshow(img_rot02)
    plt.title(angle_neg)
    plt.axis('off');
    
    i+=1

In [ ]:
def image_generator(input_data,
                    output_data,
                    batch_size=25,
                    img_channels=3,
                    rotation_range=60,
                    negative=False,
                    input_size=(100, 100),
                    equalize_hist=True):

    img_rows, img_cols = input_size

    while True:
        
        batch_files = np.random.choice(input_data, batch_size, replace=True)
        
        if negative:
            start_range = -1 * rotation_range
        else:
            start_range = 0
            
        angle_set = np.arange(start_range,rotation_range,5)
        
        batch_images = []
        batch_labels = []

        for current_file in batch_files:

            rotation_angle = np.random.choice(a = angle_set)
            current_image = get_image(img_path = current_file)
            
            current_label = get_label(img_path=current_file, out=output_data)
            new_angle = current_label+rotation_angle
            
            
            if equalize_hist:
                current_image = equalize_adapthist(image=current_image)
                
            if new_angle>180:
                new_angle-=180

            current_image = resize(image=current_image, output_shape=(img_rows,img_cols), mode='reflect')
            current_image = rotate(image=current_image, angle=rotation_angle, mode='reflect')
            
            batch_images.append(current_image)
            batch_labels.append(new_angle)

        batch_images = np.array(batch_images)
        batch_labels = np.array(batch_labels)
        
        batch_images = np.expand_dims(batch_images,-1)
        batch_labels = np.expand_dims(np.expand_dims(np.expand_dims(batch_labels,1),1),1)
        
        batch_x, batch_y = batch_images, batch_labels

        yield (batch_x, batch_y)

In [ ]:
from keras import backend as K
from keras.models import Model
from keras.layers import Input, BatchNormalization, Conv2D, Dense, Activation, Dropout

In [ ]:
img_size = (100,100,1)
dropout_rate = 0.25

In [ ]:
from keras.regularizers import l2

In [ ]:
dropout_rate

In [ ]:
K.clear_session()

main_input = Input(shape=img_size)

x = BatchNormalization()(main_input)
x = Conv2D(filters=16, kernel_size=(8,8),kernel_regularizer=l2(l=1))(x)
x = Conv2D(filters=16, kernel_size=(8,8),kernel_regularizer=l2(l=1))(x)

x = BatchNormalization()(x)
x = Conv2D(filters=32, kernel_size=(6,6),strides=(2,2),kernel_regularizer=l2(1))(x)
x = Conv2D(filters=32, kernel_size=(6,6),strides=(2,2),kernel_regularizer=l2(1))(x)
x = Activation('relu')(x)
x = Dropout(dropout_rate)(x)

x = BatchNormalization()(x)
x = Conv2D(filters=64, kernel_size=(4,4),strides=(2,2),kernel_regularizer=l2(1))(x)
x = Conv2D(filters=64, kernel_size=(4,4),strides=(2,2),kernel_regularizer=l2(1))(x)
x = Activation('relu')(x)
x = Dropout(dropout_rate)(x)

x = BatchNormalization()(x)
x = Conv2D(filters=128, kernel_size=(2,2),strides=(1,1),kernel_regularizer=l2(1))(x)
x = Conv2D(filters=128, kernel_size=(2,2),strides=(1,1),kernel_regularizer=l2(1))(x)
x = Activation('relu')(x)
x = Dropout(dropout_rate)(x)

x = Dense(units=1,kernel_regularizer=l2(l=1))(x)
main_output = Activation('relu')(x)

model = Model(inputs=[main_input], outputs=[main_output])

model.summary()

In [ ]:
angle_range = 30

In [ ]:
train_generator = image_generator(data_files_zero[:-10],data_labels,rotation_range=angle_range, negative=True)
test_generator = image_generator(data_files_zero[-10:],data_labels,rotation_range=angle_range, negative=True)

In [ ]:
from keras.callbacks import ModelCheckpoint,EarlyStopping

MC = ModelCheckpoint(filepath='./02.model/model_03.h5')
ES = EarlyStopping(patience=10)

In [ ]:
from keras.optimizers import Adam

In [ ]:
model.compile(optimizer=Adam(lr=1e-4),loss='mse')
model.fit_generator(generator = train_generator,
                    validation_data = test_generator,
                    callbacks = [MC,ES],
                    steps_per_epoch = 100,
                    validation_steps = 10,
                    epochs = 100,
                    verbose = 2)

In [ ]:
from keras.models import load_model

model = load_model('./02.model/model_03.h5')

In [ ]:
from tqdm import tqdm

In [ ]:
images_actual = []
y_actual = []
y_predicted = []

for i in tqdm(range(100)):
    
    x,y_real = test_generator.next()
    y_predict = model.predict_on_batch(x)
    
    images_actual+=[x]
    y_actual += [y_real]
    y_predicted += [y_predict]

images_actual = np.concatenate(images_actual)
y_real = np.concatenate(y_actual)
y_predicted = np.concatenate(y_predicted)

In [ ]:
def two_scales(ax1, x, data1, data2, c1, c2):
    """

    Parameters
    ----------
    ax : axis
        Axis to put two scales on

    x : array-like
        x-axis values for both datasets

    data1: array-like
        Data for left hand scale

    data2 : array-like
        Data for right hand scale

    c1 : color
        Color for line 1

    c2 : color
        Color for line 2

    Returns
    -------
    ax : axis
        Original axis
    ax2 : axis
        New twin axis
    """
    ax2 = ax1.twinx()

    ax1.scatter(x, data1, color=c1, alpha=0.5, s=10)
    ax1.set_xlabel('$Y$ ( Actual )')
    ax1.set_ylabel('$\hat{Y}$ ( Predicted)',rotation=0)

    ax2.scatter(x, data2, color=c2, alpha=0.5, s=5)
    ax2.set_ylabel('$\Delta$ ( Error )',rotation=0)
    
    return ax1, ax2

In [ ]:
images_actual = np.squeeze(images_actual,axis=-1)
y_real = np.squeeze(np.squeeze(y_real,axis=-1))
y_predicted = np.squeeze(np.squeeze(y_predicted,axis=-1))

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
x_c = y_real
y_1 = y_predicted
y_2 = y_real-y_predicted

fig, ax = plt.subplots(figsize=(8,4), dpi=100)
ax1, ax2 = two_scales(ax1 = ax, 
                      x = x_c, 
                      data1 = y_1, 
                      data2 = y_2, 
                      c1 = 'r', 
                      c2 = 'b')


# Change color of each axis
def color_y_axis(ax, color):
    """Color your axes."""
    for t in ax.get_yticklabels():
        t.set_color(color)
    return None

print(
    r2_score(y_real, y_predicted),
    np.sqrt(mean_squared_error(y_real, y_predicted))
)

color_y_axis(ax1, 'r')
color_y_axis(ax2, 'b')

In [ ]:
np.sqrt(mean_squared_error(y_real, y_predicted))

In [ ]:
cutoff = 10
image_set = images_actual[np.absolute(y_2)>cutoff]
large_devs = pd.DataFrame(
                        {'Actual':x_c[np.absolute(y_2)>cutoff],
                         'Delta':y_1[np.absolute(y_2)>cutoff],
                         'Error':y_2[np.absolute(y_2)>cutoff]
                        }
                        )
large_devs.sort_values('Error',ascending=False)

In [ ]:
plt.subplot(121)
plt.imshow(image_set[-5]);

plt.subplot(122)
plt.imshow(image_set[-2]);

In [ ]:
weights = model.get_weights()

In [ ]:
focus_layer = weights[10]
focus_layer.shape

In [ ]:
hist_indices = []

for i in range(len(weights)):
    x = weights[i]
    if len(x.shape)>1:
        print(x.shape )
        hist_indices+=[i]
        
hist_indices

In [ ]:
plt.figure(figsize=(16,7))

for i in range(len(hist_indices)):
    
    weight_index = hist_indices[i]
    focus_layer = weights[weight_index]
    
    plt.subplot(2,5,i+1)
    plt.hist(focus_layer.flatten(),bins=100);